# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandini3206/flyrank-ml-workspace/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 2: Refresh / Content Opportunity Scoring** is best framed as a **Ranking and Scoring** task.

While we can train a binary classifier to predict whether a page's performance will decline (`is_declining_label`), the ultimate business goal is not a yes/no prediction. Instead, we want to produce a prioritized queue. By outputting continuous probability scores from our machine learning model (such as the probability that a page will decline) and combining it with potential opportunity/visibility metrics (like `impressions_90d`), we can rank the entire content inventory. Content editors and SEO managers can then work through this ranked queue from top to bottom, focusing their limited review capacity on the highest-priority, high-impact pages.

In [ ]:
# Verify class distribution for the proxy label in the dataset
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

total = len(df)
declining = (df['trend_direction'] == 'down').sum()
stable_up = total - declining

print(f"Total pages for scoring: {total}")
print(f"Declining pages (Class 1): {declining} ({declining/total*100:.2f}%)")
print(f"Stable/Up/Flat pages (Class 0): {stable_up} ({stable_up/total*100:.2f}%)")

## 2. Target or proxy

Our target is `is_declining_label`, which is a binary proxy label derived from `trend_direction == 'down'`.

- **Where does it come from:** It is derived from `trend_pct`, which measures the percentage change in GSC search impressions between the most recent 30-day window and the preceding 30-day window (days 31–60 back). If the drop is greater than 20% (`trend_pct < -20.0`), `trend_direction` is set to `'down'`, and `is_declining_label` is 1; otherwise, it is 0.
- **Why it is a proxy:** It is an observed historical outcome within our 90-day snapshot, rather than a hand-defined arbitrary rule. However, because it represents a retrospective trend calculated over the same broad 90-day window, we must treat it as a proxy for *future* decline. In a full production system, we would predict decline in a future, completely non-overlapping time window (e.g., features from prior 90 days, target from the next 30 days) to prevent any lookahead bias or data leakage.

In [ ]:
# Show how the proxy label is derived from the trend_pct distribution
import numpy as np

# Verify that all 'down' trend_direction matches trend_pct < -20%
down_subset = df[df['trend_direction'] == 'down']
max_trend_pct_down = down_subset['trend_pct'].max()
print(f"Maximum trend_pct for declining pages: {max_trend_pct_down:.1f}%")

# Double check that we have no overlap
non_down_subset = df[df['trend_direction'] != 'down']
min_trend_pct_non_down = non_down_subset['trend_pct'].min()
print(f"Minimum trend_pct for stable/up/flat pages: {min_trend_pct_non_down:.1f}%")

## 3. Success metric

We choose **Precision@K** (specifically **Precision@50**) as our primary, business-defensible evaluation metric, alongside **Average Precision (AP)**.

- **Why Precision@50:** Content editors have a fixed, limited operational capacity—for example, they might only have time to review and update 50 pages per week. Therefore, we care deeply about the accuracy of the top 50 recommendations we hand them. If Precision@50 is 74% (as achieved by the random forest baseline), it means that 37 out of the 50 pages recommended for refresh are indeed declining, minimizing wasted editorial hours on healthy pages.
- **Why AP:** Average Precision summarizes the entire precision-recall curve, ensuring that high-probability candidates are ranked near the top of the queue across the whole portfolio, regardless of the specific cutoff capacity.

In [ ]:
# Let's verify our baseline precision metrics if we were to select pages based on a simple freshness heuristic
# e.g., ranking purely by days_since_last_update (oldest first) for pages with some minimum visibility
visible_pages = df[df['impressions_90d'] >= 500].copy()
top_50_stale = visible_pages.sort_values(by='days_since_last_update', ascending=False).head(50)
precision_at_50_stale = (top_50_stale['trend_direction'] == 'down').mean()

print(f"Number of visible pages (>= 500 impressions): {len(visible_pages)}")
print(f"Precision@50 when ranking purely by oldest updated pages: {precision_at_50_stale:.2f}")

## 4. The unit of analysis, as a real dataframe

The **unit of analysis (grain)** is a **single unique page (content item)** for a given client site.

- **One row equals:** One unique `content_id` associated with a specific `client_id`, containing historical trailing 90-day performance signals (impressions, clicks, sessions, engagement metrics, word count, age, etc.).
- **Dataframe Representation:** Below we load the raw starter dataset to inspect the grain, dimensions, and schema.

In [ ]:
# Load the raw slice and display its shape, columns, and grain validation
print(f"Dataframe Shape: {df.shape}")
print("\nFirst 3 rows of key content and performance features:")
print(df[['content_id', 'client_id', 'content_type', 'impressions_90d', 'days_since_last_update', 'trend_direction']].head(3))

# Verify that content_id is the unique primary key (grain check)
is_unique = df['content_id'].nunique() == len(df)
print(f"\nIs content_id unique per row (validating grain)? {is_unique}")

## 5. Why ML beats a fixed rule here

A simple, hand-crafted rule (e.g., `days_since_last_update >= 180 and impressions_90d >= 500`) is too rigid and cannot scale to meet complex organic search dynamics.

1. **Hard Cutoffs / Threshold Boundaries:** A page with 499 impressions or updated 179 days ago is completely ignored by a hard cutoff rule, even if its traffic is collapsing rapidly. ML provides smooth, continuous probability estimates that prevent these boundary errors.
2. **Multi-dimensional Signal Interaction:** Search decay is a tangled interaction of search volume, impressions, CTR, average position, scroll rates, word count, and client-level traffic baseline. Writing an if-statement to handle non-linear relationships across 20+ signals without conflict is practically impossible and becomes completely unmaintainable.
3. **Capacity Matching:** Fixed rules return a highly variable number of pages week-to-week (sometimes 10, sometimes 500). ML ranks the entire portfolio continuously, allowing editorial teams to always extract exactly the top K pages that match their current operational capacity.

In [ ]:
# Let's count how many pages would trigger a simple fixed rule versus continuous scoring
# Rule: stale and highly visible pages
rule_stale_visible = (df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)
count_rule = rule_stale_visible.sum()

print(f"Number of pages triggering our hard-cutoff baseline rule: {count_rule}")
print(f"Of these, how many are actually declining? {(df[rule_stale_visible]['trend_direction'] == 'down').sum()} ({df[rule_stale_visible]['trend_direction'] == 'down'].mean()*100:.1f}%)")

# Compare this to total declining pages with high visibility that the rule completely misses:
missed_declining = ((df['trend_direction'] == 'down') & (df['impressions_90d'] >= 500) & (~rule_stale_visible)).sum()
print(f"Number of highly visible declining pages that the rigid rule completely misses: {missed_declining}")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.